In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

import sys
from pathlib import Path
sys.path.append(str(Path("../src").resolve()))
from quantlab.data.binance import load_parquet
from quantlab.features.technical_features import add_returns, add_momentum, add_sma
from quantlab.backtest.costs import apply_transaction_costs
from quantlab.backtest.metrics import equity_curve, max_drawdown

%load_ext autoreload
%autoreload 2

In [ ]:
df = load_parquet('../data/raw/BTCUSDT_1d.parquet')
df = add_momentum(df, 20)
df = add_returns(df)
df = add_sma(df, 20)

## Risk analysis

A trading strategy is not fully described by its expected return. Two strategies may have the same average return while exposing the investor to very different kinds of losses.

Let $R_t=(r_1,r_2,\dots,r_t)$ denote the time series of the returns of a strategy over period $t$. From a risk-management perspective, we are interested in the entire distribution of $R_t$, not only in its mean.

The mean $\mathbb E[R_t]$ describes the average return, while the dispersion and shape of the distribution describe the uncertainty around that return. In particular, relevant questions include:

* How variable are the returns?
* How often do large losses occur?
* How severe are losses in the worst cases?
* Does risk remain stable over time?
* How much can the portfolio decline from a previous peak?

### Why volatility is not enough

Volatility is usually defined as the standard deviation of returns: $\sigma_{R_t}=\sqrt{\operatorname{Var}(R_t)}$.

It measures the typical magnitude of fluctuations around the average return. However, volatility treats positive and negative deviations symmetrically. A large positive return increases volatility just as a large negative return does, even though only the latter represents a loss.

Moreover, volatility alone does not describe the tails of the return distribution. Two strategies can have the same volatility while having very different probabilities of extreme losses.

### Why the Sharpe ratio is not enough

The annualized Sharpe ratio compares expected excess return with return volatility: 
$
\operatorname{Sharpe} = \sqrt{260}\frac{\mathbb E[R_t-R_{f,t}]}
{\operatorname{Std}(R_t-R_{f,t})}
$
where $R_{f,t}$ is the return of a risk-free asset. Assume the annual risk-free rate is $3%$, then the daily risk-free is $R_{f,t} = 0.03/260\approx 1.2\cdot10^{-4}$. Bitcoin average daily return is $\approx 10^{-2}$, so $R_{f,t}$ is negligible, hence the operative definition of Sharpe is
$$
\operatorname{Sharpe} = \sqrt{260}\frac{\mathbb E[R_t]}
{\operatorname{Std}(R_t)}.
$$

It measures return per unit of volatility and is useful for comparing strategies on a risk-adjusted basis.

However, the Sharpe ratio compresses the whole return distribution into only two quantities: mean and standard deviation. It does not distinguish between strategies with symmetric returns and strategies with rare but severe crashes. It also does not describe when losses occur or how long it takes to recover from them.

### Why maximum drawdown is not enough

Let $v_s=(r_2-1)(r_3-1)\cdots(r_s-1)$ and $V_t=\left(v_2,v_3,\dots,v_t\right)$ be the equity curve of the strategy and let
$$
M_t=\max_{s\le t}V_s
$$
be the highest equity value observed up to time $t$. The running percentage drawdown is

$$
D_t=\frac{V_t}{M_t}-1.
$$

The maximum drawdown is

$$
\operatorname{MDD}=\min_t D_t.
$$

It measures the largest historical peak-to-trough percentage decline.

Maximum drawdown is intuitive and economically meaningful, but it depends on a single worst historical episode. It does not describe the frequency of losses, the overall distribution of returns, or what could happen beyond the specific sample observed.

### Complementary risk measures

For these reasons, risk analysis normally combines several measures:

* **Rolling volatility** describes how the typical magnitude of returns changes through time.
* **Value at Risk (VaR)** estimates a loss threshold exceeded only with a chosen small probability.
* **Expected Shortfall (ES)** measures the average loss conditional on being beyond the VaR threshold.
* **Drawdown analysis** studies realized losses relative to previous equity peaks.

No single metric gives a complete description of risk. Together, these measures provide information about typical fluctuations, tail losses, changing market conditions, and historically realized declines.


### Rolling volatility

Rolling volatility estimates how the variability of daily returns changes through time. For a 30-day window,

\begin{align*}
\hat{\sigma}_t^{(30)}
=
\left[
\operatorname{Std}(r_{1},\ldots,r_{30}),\right.\\
\operatorname{Std}(r_{2},\ldots,r_{31}),\\
\operatorname{Std}(r_{3},\ldots,r_{32}),\\
\vdots\\
\operatorname{Std}(r_{t-30},\ldots,r_{t-1}),\\
\left.\operatorname{Std}(r_{t-29},\ldots,r_{t})\right]
\end{align*}


Financial returns often exhibit  **volatility clustering**: periods of large fluctuations tend to be followed by other volatile periods, while calm periods tend to persist. Rolling volatility helps identify these changing risk regimes.

In [ ]:
rolling_vol_30 = df['returns'].rolling(30).std()

In [ ]:
fig, axes = plt.subplots(2, figsize = (10,4), sharex=True)

axes[0].plot(rolling_vol_30)
axes[0].set_title('30-day rolling volatility')

axes[1].plot(df['returns'])
axes[1].set_title('Daily returns')

for ax in axes:
    ax.grid(True)

### Value at Risk (VaR)

Given $\alpha\in(0,1)$, the **VaR at $\alpha$** is a return denoted by $\operatorname{VaR}_{\alpha}$ such that daily losses won't exceed $\operatorname{VaR}_{\alpha}$ with probability at least $\alpha$ (confidence level).

Equivalently, with probability $\alpha$, daily return is $\ge-\operatorname{VaR}_{\alpha}$.

It is defined as
$$
\operatorname{VaR}_{\alpha}:=-\min\left(\operatorname{quantile}_{1-\alpha}(R_t), 0\right)
$$
(we take the $1-\alpha$ quantile of the returns only when it's a loss).

In [ ]:
def historical_var(returns, confidence_level):
    q = returns.dropna().quantile(1 - confidence_level)
    return -min(0, q)

In [ ]:
plt.figure(figsize=(8,6))

returns = df['returns'].dropna()
var_95 = historical_var(returns, 0.95)

returns.hist(bins=50)
plt.axvline(-var_95, color = 'r', linestyle='--', alpha=0.75, label='VaR at 95%: {}'.format(-round(var_95,2)))

plt.title('Returns distribution and VaR at 95%')
plt.legend()

print("\nWith at least 95% confidence, there have not been daily losses greater than {}%\n".format(round(var_95 * 100,2)))

### Expected Shortfall

The Expected Shortfall at $\alpha\%$ level measures the expected return of the portfolio in the worst $100 - \alpha\%$ of cases. Namely:
$$
\operatorname{ES}_{\alpha}:=-\Bbb E[R_t|R_t\le\operatorname{quantile}_{1-\alpha}(R_t)]
$$

Consider for example a strategy having historical returns with
$$
\operatorname{VaR}_{0.95}=3\%\hspace{1cm}\operatorname{ES}_{0.95}=7\%\;.
$$
This means that in $95\%$ of the cases, daily losses won't be worse than $3%$, but in the worst $5\%$, losses are on average of $7\%$.

We might have two different strategies, both with $\operatorname{VaR}_{0.95}=3\%$ but $\operatorname{ES}_{0.95}=4\%$ for the first and $9\%$ for the second, hence ES gives meaningful information.

In [ ]:
def expected_shortfall(returns, confidence_level):
    returns = returns.dropna()
    q = returns.quantile(1 - confidence_level)
    tail_returns = returns[returns <= q]
    if tail_returns.empty:
        return 0
    return max(-tail_returns.mean(), 0.0)

### Running Drawdown


The running drawdown measures the percentage distance between the current equity and its historical peak. It equals zero whenever the equity reaches a new high and remains negative until that peak is recovered. Its depth measures the severity of the decline, while its duration measures how long the portfolio remains underwater.

Each time the last maximum is recovered, the RD goes back to zero, and the time it does stay below zero tells the last maximum has not been recovered.

The equity curve only tells how much wealth has been accumulated. The running drawdown describes the cost of achieving that performance: at every point in time it measures how far the portfolio is below its previous historical peak. Two strategies with the same final return may exhibit completely different drawdown profiles, making one psychologically and economically (investors might withdraw theit money if the strategy is losing, or if you are leveraging 5X a drawdown of 20% makes you lose all, or even before that having problems of margin call, or even risk constraints, which if violeted forces you to change strategy) much harder to hold than the other.

In [ ]:
equity = equity_curve(df)
running_drawdown = equity / equity.cummax() - 1

In [ ]:
fig, axes = plt.subplots(2, figsize = (12,6), sharex = True)

axes[0].plot(equity)
axes[0].grid(True)
axes[0].set_title('returns equity curve')

axes[1].plot(running_drawdown)
axes[1].grid(True)
axes[1].set_title('running drawdown')

### Risk comparison across strategies

In [ ]:
from quantlab.backtest.metrics import historical_var, expected_shortfall, running_drawdown
from quantlab.portfolio.allocation import portfolio_returns
from quantlab.backtest.report import performance_summary

In [ ]:
# signals
momentum_20d_signal = pd.Series(np.where(df['momentum_20'] > 0, 1, -1), index = df.index)
momentum_moving_average_signal = pd.Series(np.where(df['close'] > df['sma_20'], 1, -1), index=df.index)

# strategies
momentum_20d_return = momentum_20d_signal.shift(1) * df['returns']
momentum_20d_return_net = apply_transaction_costs(momentum_20d_return, momentum_20d_signal, cost_per_trade=0.001)

momentum_moving_average_return = momentum_moving_average_signal.shift(1) * df['returns']
momentum_moving_average_return_net = apply_transaction_costs(momentum_moving_average_return, momentum_moving_average_signal, cost_per_trade=0.001)

equal_weight_strat = pd.concat([momentum_20d_return_net, momentum_moving_average_return_net], axis=1)
equal_weight_strat.columns = ['20d', 'moving_average_20d']
weights = pd.Series([0.5, 0.5], index=equal_weight_strat.columns)

equal_weight_strat_returns = portfolio_returns(equal_weight_strat, weights)

In [ ]:
summaries = pd.DataFrame({
    'buy_and_hold': performance_summary(pd.DataFrame({'returns':df['returns']})),
    'momentum_20': performance_summary(pd.DataFrame({'returns': momentum_20d_return_net})),
    'momentum_sma_20': performance_summary(pd.DataFrame({'returns': momentum_moving_average_return_net})),
    'equal_weight_strat': performance_summary(pd.DataFrame({'returns' : equal_weight_strat_returns}))
}).T

summaries.style.format({
    "total_return": "{:.2%}",
    "annualized_return": "{:.2%}",
    "annualized_volatility": "{:.2%}",
    "sharpe": "{:.2f}",
    "var_95": "{:.2%}",
    "exp_shortfall_95": "{:.2%}",
    "max_drawdown": "{:.2%}",
})

In [ ]:
buy_and_hold_equity_curve = equity_curve(pd.DataFrame(df['returns']))
momentum_20_equity_curve = equity_curve(pd.DataFrame({'returns': momentum_20d_return_net}))
momentum_sma_20_equity_curve = equity_curve(pd.DataFrame({'returns' : momentum_moving_average_return_net}))
equal_weight_strat_equity_curve = equity_curve(pd.DataFrame({'returns' : equal_weight_strat_returns}))


buy_and_hold_running_drawdown = running_drawdown(pd.DataFrame(df['returns']))
momentum_20_running_drawdown = running_drawdown(pd.DataFrame({'returns': momentum_20d_return_net}))
momentum_sma_20_running_drawdown = running_drawdown(pd.DataFrame({'returns' : momentum_moving_average_return_net}))
equal_weight_strat_running_drawdown = running_drawdown(pd.DataFrame({'returns' : equal_weight_strat_returns}))

In [ ]:
fig, axes = plt.subplots(2, figsize=(13,8), sharex=True)

axes[0].plot(buy_and_hold_equity_curve, label = 'buy_and_hold')
axes[0].plot(momentum_20_equity_curve, label = 'momentum_20')
axes[0].plot(momentum_sma_20_equity_curve, label = 'momentum_sma_20')
axes[0].plot(equal_weight_strat_equity_curve, label = 'equal_weight_strat')
axes[0].set_title('Strategies\' Equity Curves')
axes[0].grid(True)             
axes[0].legend()


axes[1].plot(buy_and_hold_running_drawdown, label = 'buy_and_hold')
axes[1].plot(momentum_20_running_drawdown, label = 'momentum_20')
axes[1].plot(momentum_sma_20_running_drawdown, label = 'momentum_sma_20')
axes[1].plot(equal_weight_strat_running_drawdown, label = 'equal_weight_strat')
axes[1].set_title('Strategies\' Drawdown Curves')
axes[1].grid(True)             
axes[1].legend()